# Random Forest : dataset [AMES](http://jse.amstat.org/v19n3/decock.pdf)

## Data

We pick up where we left off after data mining: we need to retrieve the dataset again, and pre-process it.

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-ames.git
!ls -l dataset-ames/

In [ ]:
!wc -l dataset-ames/train.csv

In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import sklearn.ensemble
import sklearn.model_selection
import scipy.stats
import warnings

warnings.filterwarnings("ignore")


def preprocess(train_file, test_file):
  train_X = pandas.read_csv(train_file, index_col="Id")
  test_X = pandas.read_csv(test_file, index_col="Id")

  train_y = train_X.pop("SalePrice")

  all_X = pandas.concat([train_X, test_X])

  # Fill with median
  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  # Fill with mode
  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  # Fill with 0
  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  # Other fills
  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  # Anything else is set to the String "NA"
  all_X = all_X.fillna("NA")

  # Numeric coded categories need to be cast as String to be interpreted as
  # categories and not numerical values.
  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"Nombre de NAs : {all_X.isnull().sum().sum()}")

  dummies = pandas.get_dummies(all_X)
  return (
    dummies.iloc[: train_X.shape[0], :],
    train_y,
    dummies.iloc[train_X.shape[0] :, :],
  )

In [ ]:
train_X, train_y, test_X = preprocess("dataset-ames/train.csv", "dataset-ames/test.csv")

We have now pre-processed and visualised the data. We still need to understand how to train a model, set its hyper-parameters and extract the information that may be of interest to us.

## Output preparation

Our model of choice for this tutorial will be Random Forest, to test some of its strengths.

However, before we begin, we have one last pre-processing detail to deal with: we detected a positive skew in the output variable (`SalePrice`) without correcting it.

*Log-transform the `train_y` variable to correct its positive skew. Note that there is no point in transforming the other variables because, as you may have noticed when we detailed the current algorithms, decision trees are insensitive to monotonic transformations of their input variables*

In [ ]:
# your code here

### Solution

In [ ]:
seaborn.distplot(train_y, fit=scipy.stats.norm)
plt.title("SalePrice distribution before normalization")
plt.show()

# Output Log transformation
train_y = numpy.log1p(train_y)

seaborn.distplot(train_y, fit=scipy.stats.norm)
plt.title("SalePrice distribution after normalization")
plt.show()

## Evaluation
With our data now ready, we need to find a proper evaluation method for our models.

*Use [`cross_val_score`](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) to write a `score` function that will evaluate a model with a 5-fold cross-validation that uses [`root mean squared error`](https://scikit-learn.org/stable/modules/model_evaluation.html).*

*Note that the scoring function that can be used is ``neg_root_mean_squared_error`` because in the sklearn api, a score has the opposite behaviour to an error, so to use an error as a score, you must use the negative version of the error. (A solution with a high error must have a lower score than a solution with a low error)*.

In [ ]:
# your code here

### Solution

In [ ]:
def score(model):
  rmse = -sklearn.model_selection.cross_val_score(
    model, train_X, train_y, cv=5, scoring="neg_root_mean_squared_error"
  )
  return rmse.mean(), rmse.std()

## Training
We now have a reliable evaluation tool. We can start experimenting with advanced regression models.

*Test a [`RandomForestRegressor`](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) initialised with `n_estimators=10`: this will be a baseline to outperform.*

In [ ]:
# your code here

### Solution

In [ ]:
print(score(sklearn.ensemble.RandomForestRegressor(n_estimators=10)))

## Optimisation
One of the first things to do to get the most out of a model is to set its parameters.

*Use [`RandomizedSearchCV`](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) on the parameters `n_estimators`, `max_features`, `max_depth` and `min_samples_leaf` to find a good parameter setting.*

In [ ]:
# your code here

### Solution

In [ ]:
# max_features=None correspond to max_features=n_features (cf. doc sklearn)
rscv = sklearn.model_selection.RandomizedSearchCV(
  estimator=sklearn.ensemble.RandomForestRegressor(),
  param_distributions=dict(
    max_samples=[0.6, 0.8, 1.0],
    n_estimators=[10, 100, 500],
    max_features=["log2", "sqrt", None],
    max_depth=[8, 128],
    min_samples_leaf=[3, 5],
  ),
  n_iter=30,
  scoring="neg_root_mean_squared_error",
  cv=2,
  random_state=889,
  n_jobs=-1,
  return_train_score=True,
  verbose=10,
)

rscv.fit(train_X, train_y)

print(f"Best loss : {-rscv.best_score_}")
print(f"Best parameters : {rscv.best_params_}")

# Best model trained
rfr = rscv.best_estimator_

## Prediction

We can already make predictions with our best trained model. However, we must be careful to do the reverse transformation of the `numpy.log1p` performed earlier.

*Compute the predictions of the model on the test set using the functions [`sklearn.ensemble.RandomForestRegressor.predict`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.predict) and [`numpy.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html)*.

In [ ]:
# your code here

### Solution

In [ ]:
predictions = numpy.expm1(rfr.predict(test_X))
print(f"test set shape : {test_X.shape}")
print(f"predictions shape : {predictions.shape}")

seaborn.distplot(predictions)
plt.title("Predictions density")
plt.xlabel("Price in $")
plt.ylabel("Density")
plt.show()

## Feature importance

As this model is satisfactory for the moment, let us look at the possibilities offered by Random Forest. For example, it is possible to calculate the importance of features by observing the average fall in error (or impurity) that they cause.

*Use the `feature_importances_` attribute of your Random Forest to understand which features are most important.*

In [ ]:
# your code here

### Solution

In [ ]:
importances = pandas.DataFrame(
  {"importance": rfr.feature_importances_}, index=train_X.columns
)
importances.sort_values(by="importance", ascending=False, inplace=True)
importances.iloc[:20, :][::-1].plot.barh()
plt.show()

## Generalization performance
It is also possible to measure the generalisation of a forest on examples that its trees could not see because of the boosting process.

*Use the `oob_score_` attribute to measure the impact of the number of trees in the forest. To do this, train a model with 1 to 20 trees and plot the OOB score. The OOB score that `scikit-learn` calculates is the R² score. Good resources for interpreting it are available on stats.stackexchange: [here](https://stats.stackexchange.com/questions/70704/interpreting-out-of-bag-error-estimate-for-randomforestregressor) and [there](https://stats.stackexchange.com/questions/133406/is-a-negative-oob-score-possible-with-scikit-learns-randomforestregressor).*

In [ ]:
# your code here

### Solution

In [ ]:
oob_scores = []
max_trees = 50
for i in range(1, max_trees + 1):
  rfr.set_params(n_estimators=i, oob_score=True)
  rfr.fit(train_X, train_y)
  # Log OOB score for each number of trees in each forest
  # OOB score = R² score.
  oob_score = rfr.oob_score_
  oob_scores.append(oob_score)

plt.title("Effect of the number of trees on forest performance")
plt.xlabel("number of trees")
plt.ylabel("OOB score")
plt.plot(range(1, max_trees + 1), oob_scores)
plt.show()

In [ ]:
# Zoom on OOB score when positives (better than a constant predictor)
oob_array = numpy.array(oob_scores)
left_context = 3
first_positive = numpy.where(oob_array > 0)[0][0]
first_point = max(1, first_positive - left_context)

plt.title("Effect of the number of trees on forest performance")
plt.xlabel("number of trees")
plt.ylabel("OOB score")
plt.plot(range(first_point + 1, max_trees + 1), oob_array[first_point:])

In [ ]:
# Display the first tree with GraphViz
from sklearn.tree import export_graphviz

# Export as dot file
export_graphviz(
  rfr.estimators_[0],
  out_file="tree.dot",
  feature_names=train_X.columns,
  rounded=True,
  proportion=False,
  precision=2,
  filled=True,
)

# Convert to png using system command (requires Graphviz)
from subprocess import call

call(["dot", "-Tpng", "tree.dot", "-o", "tree.png", "-Gdpi=600"])
# Convert to svg
call(["dot", "-Tsvg", "tree.dot", "-o", "tree.svg"])

# Display in jupyter notebook
from IPython.display import Image

Image(filename="tree.png")